# ALLM - Colab Phase 1

هذا الدفتر يثبت المشروع ويشغّل بوابة المرحلة الأولى على Colab. لا تضع مفاتيح أو بيانات خاصة داخل الدفتر.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/sello830-netizen/ALLM.git"
REPO_DIR = Path("/content/ALLM")

if not REPO_DIR.exists():
    !git clone $REPO_URL $REPO_DIR
else:
    !git -C $REPO_DIR pull --ff-only
%cd $REPO_DIR

In [ ]:
%pip install -e .
!python --version
!python scripts/phase1_local_check.py --output reports/phase1-colab.json

In [ ]:
!python scripts/build_fixture_manifest.py data/manifests/fixture-colab.json
!python scripts/evaluate_fixture.py
!python scripts/run_smoke_baseline.py data/fixtures/smoke.txt runs-colab.jsonl

## Neural baseline smoke test

بعد ظهور Tesla T4 من `nvidia-smi`، شغّل الخلية التالية لإثبات دورة PyTorch الأولى. النموذج صغير وfixture للاختبار فقط.

In [ ]:
%pip install torch
!nvidia-smi
!python scripts/train_torch_baseline.py data/fixtures/smoke.txt runs-colab.jsonl artifacts/torch-tiny-baseline.pt

## Inference من checkpoint

بعد حفظ checkpoint في Google Drive، شغّل الخلية التالية لاختبار التوليد. النتيجة smoke-only لأن التدريب تم على fixture صغير.

In [ ]:
CHECKPOINT = "/content/drive/MyDrive/ALLM-artifacts/torch-tiny-baseline.pt"
!python scripts/generate_torch.py "$CHECKPOINT" "هذه" --max-new-tokens 20

## Original-only synthetic pilot

هذه التجربة تقيس baseline على النص التجريبي قبل إضافة أي views. التقسيم يتم على مستوى الوثيقة وبـseed ثابت.

In [ ]:
!python scripts/prepare_text_splits.py \
  data/fixtures/arabic_multiview_pilot_001.json \
  data/fixtures/arabic_multiview_pilot_001_split

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/ALLM-artifacts

!python scripts/train_torch_baseline.py \
  data/fixtures/arabic_multiview_pilot_001_split/train.txt \
  /content/drive/MyDrive/ALLM-artifacts/runs-original-pilot.jsonl \
  /content/drive/MyDrive/ALLM-artifacts/torch-original-pilot.pt \
  --dev-input data/fixtures/arabic_multiview_pilot_001_split/dev.txt \
  --max-steps 100 \
  --batch-size 4

In [ ]:
!python scripts/generate_torch.py \
  /content/drive/MyDrive/ALLM-artifacts/torch-original-pilot.pt \
  "بدأت الشركة" \
  --max-new-tokens 30

## Multi-View pilot

نبني ثلاث حالات: original، one_view_per_source، وmultiview. نثبت architecture وseed و500 steps وbatch=4، ونسجل token budgets لأنها ليست متساوية.

In [ ]:
!python scripts/build_multiview_pilot.py \
  data/fixtures/arabic_baseline_corpus_500.jsonl \
  data/fixtures/arabic_baseline_corpus_500_multiview \
  --seed 17

In [ ]:
!python scripts/train_torch_baseline.py \
  data/fixtures/arabic_baseline_corpus_500_multiview/one_view/train.txt \
  /content/drive/MyDrive/ALLM-artifacts/runs-500-one-view.jsonl \
  /content/drive/MyDrive/ALLM-artifacts/torch-500-one-view.pt \
  --dev-input data/fixtures/arabic_baseline_corpus_500_multiview/one_view/dev.txt \
  --max-steps 500 \
  --batch-size 4

In [ ]:
!python scripts/train_torch_baseline.py \
  data/fixtures/arabic_baseline_corpus_500_multiview/multiview/train.txt \
  /content/drive/MyDrive/ALLM-artifacts/runs-500-multiview.jsonl \
  /content/drive/MyDrive/ALLM-artifacts/torch-500-multiview.pt \
  --dev-input data/fixtures/arabic_baseline_corpus_500_multiview/multiview/dev.txt \
  --max-steps 500 \
  --batch-size 4

## البيانات الخاصة

لبيانات التدريب المرخّصة أو الخاصة، استخدم Google Drive أو Secret/Storage مناسبًا. لا ترفع corpus خاصًا إلى مستودع GitHub أو إلى مخرجات Notebook العامة.

In [ ]:
# اختياري: استخدمه فقط عند الحاجة إلى بيانات محفوظة في Drive.
# from google.colab import drive
# drive.mount('/content/drive')
# PRIVATE_DATA_DIR = '/content/drive/MyDrive/ALLM-data'